# Flow Matching on 32x32 Mazes

This notebook trains the same timestep-conditioned U-Net as the DDPM experiment, but with conditional flow matching. A straight interpolation connects Gaussian noise at `t=0` to the embedded maze at `t=1`; the network predicts the path velocity. Sampling integrates the learned ODE from noise to data.

The maze is represented by two fixed orthonormal pixel embeddings (`wall=0`, `passage=1`). The final continuous image is greedily decoded by dot product. Run cells manually; training may take a while.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

import importlib
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import clear_output, display
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from plotly.subplots import make_subplots

from _shared.maze_data import generate_dataset, solvability_rate
import _shared.model as model_module

importlib.reload(model_module)
from _shared.model import (
    Denoiser,
    decode_probabilities,
    embed_binary,
    make_binary_embeddings,
)

SEED = 7
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project: {PROJECT_ROOT} | device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} | CUDA runtime: {torch.version.cuda}")
else:
    print(
        f"CUDA unavailable to PyTorch (torch.version.cuda={torch.version.cuda}). Run uv sync and select the project .venv kernel."
    )

Project: c:\Users\shich\Src\discrete_diffusion | device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU | CUDA runtime: 12.8


## 1. Synthetic Dataset

Randomized DFS generates perfect mazes. `0` means wall; `1` means passage.

In [2]:
N_MAZES = 12_000
VAL_FRACTION = 0.1
mazes = generate_dataset(N_MAZES, seed=SEED)
n_val = int(N_MAZES * VAL_FRACTION)
train_mazes = torch.from_numpy(mazes[:-n_val].copy()).long()
val_mazes = torch.from_numpy(mazes[-n_val:].copy()).long()
print(
    "all:", mazes.shape, "| train:", len(train_mazes), "| validation:", len(val_mazes)
)
print(f"Generated-maze solvability: {solvability_rate(mazes):.1%}")

fig = go.Figure(
    go.Heatmap(
        z=mazes[0],
        zmin=0,
        zmax=1,
        colorscale=[[0, "black"], [1, "white"]],
        showscale=False,
    )
)
fig.update_layout(
    title="Synthetic maze example",
    width=420,
    height=420,
    yaxis=dict(autorange="reversed", scaleanchor="x"),
)
fig.show()

all: (12000, 32, 32) | train: 10800 | validation: 1200
Generated-maze solvability: 100.0%


## 2. Forward Flow Path

For a clean embedded maze `x_data` and independent Gaussian noise `z`, define the conditional path

`x_t = (1 - t) * z + t * x_data`, with `t` sampled uniformly from `[0, 1]`.

Its target velocity is constant along the path: `v_target = x_data - z`. This panel decodes the continuous states using the same fixed-temperature embedding dot-product visualization at every time; the visualization does not alter the path.

In [3]:
EMBEDDING_DIM = 2
TEMPERATURE = 0.5
binary_embeddings = make_binary_embeddings(EMBEDDING_DIM, DEVICE)
print("Fixed unit embeddings [wall=0, passage=1]:")
print(binary_embeddings.detach().cpu())
print(
    f"Embedding cosine similarity: {torch.dot(binary_embeddings[0], binary_embeddings[1]).item():.2e}"
)

preview_maze = torch.from_numpy(mazes[0].copy()).long().unsqueeze(0).to(DEVICE)
x_data = embed_binary(preview_maze, binary_embeddings)
preview_noise = torch.randn_like(x_data)
preview_times = [0.0, 0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9, 1.0]
with torch.no_grad():
    preview_frames = []
    for t in preview_times:
        t_batch = torch.full((1,), t, device=DEVICE)
        x_t = (1.0 - t_batch[:, None, None, None]) * preview_noise + t_batch[
            :, None, None, None
        ] * x_data
        probabilities = decode_probabilities(x_t, binary_embeddings, TEMPERATURE)
        preview_frames.append(probabilities[0, 1].cpu().numpy())

path_fig = make_subplots(
    rows=1,
    cols=len(preview_times),
    subplot_titles=[f"t={t:g}" for t in preview_times],
)
for col, frame in enumerate(preview_frames, start=1):
    path_fig.add_trace(
        go.Heatmap(
            z=frame,
            zmin=0,
            zmax=1,
            colorscale=[[0, "black"], [1, "white"]],
            showscale=False,
        ),
        row=1,
        col=col,
    )
path_fig.update_layout(
    title="Linear flow path: black = p(0), white = p(1)",
    width=1700,
    height=360,
)
path_fig.update_xaxes(showticklabels=False)
path_fig.update_yaxes(showticklabels=False, autorange="reversed")
path_fig.show()

Fixed unit embeddings [wall=0, passage=1]:
tensor([[1., 0.],
        [0., 1.]])
Embedding cosine similarity: 0.00e+00


## 3. Model Architecture

The U-Net structure is shared with the DDPM notebook: input shape `(B, 2, 32, 32)`, base channels `64` with encoder widths `64 -> 128 -> 256`, residual blocks conditioned by a `128`-dimensional time embedding, and a two-channel output. Here the output represents velocity `v(x_t, t)`, not Gaussian noise. The normalized flow time is scaled to `[0, 100]` before sinusoidal embedding so the existing time encoder receives a useful range. See `notebooks/_shared/model.py` for the implementation.

In [4]:
BASE_CHANNELS = 64
TIME_DIM = 128
TIME_EMBEDDING_SCALE = 100.0

model = Denoiser(
    embedding_dim=EMBEDDING_DIM,
    base_channels=BASE_CHANNELS,
    time_dim=TIME_DIM,
).to(DEVICE)
print(model)
print(
    f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

Denoiser(
  (time_mlp): Sequential(
    (0): Linear(in_features=128, out_features=512, bias=True)
    (1): SiLU()
    (2): Linear(in_features=512, out_features=128, bias=True)
  )
  (input): Conv2d(2, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (enc32): ResBlock(
    (norm1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (time_projection): Linear(in_features=128, out_features=64, bias=True)
    (norm2): GroupNorm(8, 64, eps=1e-05, affine=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (skip): Identity()
  )
  (down16): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (enc16): ResBlock(
    (norm1): GroupNorm(8, 128, eps=1e-05, affine=True)
    (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (time_projection): Linear(in_features=128, out_features=128, bias=True)
    (norm2): GroupNorm(8, 128, eps=1e-05, a

## 4. Flow Matching Training

For each maze, sample `z ~ N(0, I)` and `t ~ Uniform(0, 1)`, then form `x_t = (1 - t)z + t x_data`. Train the U-Net to predict `x_data - z` with MSE. Training samples one point on the path directly; it does not numerically integrate or unroll the ODE. Validation uses the same objective. The live Plotly loss chart updates in place.

In [5]:
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 2e-4
PLOT_EVERY = 100

train_loader = DataLoader(
    TensorDataset(train_mazes), batch_size=BATCH_SIZE, shuffle=True, drop_last=True
)
val_loader = DataLoader(TensorDataset(val_mazes), batch_size=BATCH_SIZE, shuffle=False)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


def batch_loss(maze_batch):
    x_data = embed_binary(maze_batch.to(DEVICE), binary_embeddings)
    noise = torch.randn_like(x_data)
    t = torch.rand(x_data.shape[0], device=DEVICE)
    t_image = t[:, None, None, None]
    x_t = (1.0 - t_image) * noise + t_image * x_data
    target_velocity = x_data - noise
    predicted_velocity = model(x_t, t * TIME_EMBEDDING_SCALE)
    return F.mse_loss(predicted_velocity, target_velocity)


@torch.no_grad()
def validation_loss():
    model.eval()
    values = [batch_loss(batch).item() for (batch,) in val_loader]
    model.train()
    return float(np.mean(values))


model.train()
train_history, val_history = [], []
loss_fig = go.Figure()
loss_fig.add_scatter(name="train", mode="lines")
loss_fig.add_scatter(name="validation", mode="lines+markers")
loss_fig.update_layout(
    title="Flow matching velocity MSE",
    xaxis_title="Optimization step",
    yaxis_title="MSE of predicted velocity",
)
global_step = 0
for epoch in range(EPOCHS):
    for (batch,) in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = batch_loss(batch)
        loss.backward()
        optimizer.step()
        train_history.append(loss.item())
        global_step += 1
        if global_step % PLOT_EVERY == 0:
            loss_fig.data[0].x = list(range(1, len(train_history) + 1))
            loss_fig.data[0].y = train_history
            loss_fig.data[1].x = [step for step, _ in val_history]
            loss_fig.data[1].y = [value for _, value in val_history]
            loss_fig.update_layout(
                title=f"Flow matching training | epoch {epoch + 1}/{EPOCHS} | step {global_step}"
            )
            clear_output(wait=True)
            display(loss_fig)
    epoch_train = float(np.mean(train_history[-len(train_loader) :]))
    epoch_validation = validation_loss()
    val_history.append((global_step, epoch_validation))
    loss_fig.data[0].x = list(range(1, len(train_history) + 1))
    loss_fig.data[0].y = train_history
    loss_fig.data[1].x = [step for step, _ in val_history]
    loss_fig.data[1].y = [value for _, value in val_history]
    loss_fig.update_layout(
        title=f"Flow matching training | epoch {epoch + 1}/{EPOCHS} | step {global_step}"
    )
    clear_output(wait=True)
    display(loss_fig)
    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | train={epoch_train:.4f} | val={epoch_validation:.4f}"
    )

print(f"Training complete | train={epoch_train:.4f} | val={epoch_validation:.4f}")

Epoch 30/30 | train=0.1252 | val=0.1192
Training complete | train=0.1252 | val=0.1192


## 5. Save the Model

In [6]:
ARTIFACTS = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
checkpoint_path = ARTIFACTS / "maze_flow_matching_velocity.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "binary_embeddings": binary_embeddings.detach().cpu(),
        "config": {
            "embedding_dim": EMBEDDING_DIM,
            "base_channels": BASE_CHANNELS,
            "time_dim": TIME_DIM,
            "time_embedding_scale": TIME_EMBEDDING_SCALE,
            "path": "linear_gaussian_to_data",
            "prediction_target": "velocity_x_data_minus_noise",
        },
    },
    checkpoint_path,
)
print(f"Saved: {checkpoint_path}")

Saved: c:\Users\shich\Src\discrete_diffusion\artifacts\maze_flow_matching_velocity.pt


## 6. ODE Sampling and Solvability

Sampling starts from `x(0) ~ N(0, I)` and integrates `dx/dt = v_model(x, t)` from `t=0` to `t=1` with the explicit Euler method. Intermediate snapshots show the evolving continuous state decoded with the same fixed-temperature visualization. Only the final state is greedily decoded to binary maze pixels.

In [7]:
N_METRIC_SAMPLES = 128
N_TO_SHOW = 4
ODE_STEPS = 100
PREVIEW_STEP_INDICES = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]


@torch.no_grad()
def sample_flow(model, n_samples, steps=ODE_STEPS, capture_indices=()):
    model.eval()
    x = torch.randn(n_samples, EMBEDDING_DIM, 32, 32, device=DEVICE)
    initial_noise = x.detach().clone()
    snapshots = {0: x.detach().clone()} if 0 in capture_indices else {}
    dt = 1.0 / steps

    for index in range(steps):
        t = index / steps
        times = torch.full((n_samples,), t, device=DEVICE)
        velocity = model(x, times * TIME_EMBEDDING_SCALE)
        x = x + dt * velocity
        next_index = index + 1
        if next_index in capture_indices:
            snapshots[next_index] = x.detach().clone()

    return x, snapshots, initial_noise


samples_continuous, flow_snapshots, initial_noise = sample_flow(
    model, N_METRIC_SAMPLES, capture_indices=PREVIEW_STEP_INDICES
)
print(
    "Initial Gaussian noise stats: "
    f"mean={initial_noise.mean().item():.3f}, "
    f"std={initial_noise.std().item():.3f}"
)
hard_samples = (
    torch.einsum("bdhw,kd->bkhw", samples_continuous, binary_embeddings)
    .argmax(dim=1)
    .cpu()
    .numpy()
    .astype(np.uint8)
)

fig = make_subplots(
    rows=N_TO_SHOW,
    cols=len(PREVIEW_STEP_INDICES),
    subplot_titles=[
        f"t={index / ODE_STEPS:g}"
        for _ in range(N_TO_SHOW)
        for index in PREVIEW_STEP_INDICES
    ],
)
for row in range(N_TO_SHOW):
    for col, step_index in enumerate(PREVIEW_STEP_INDICES, start=1):
        state = flow_snapshots[step_index][row : row + 1]
        probability_one = decode_probabilities(state, binary_embeddings, TEMPERATURE)[
            0, 1
        ]
        fig.add_trace(
            go.Heatmap(
                z=probability_one.cpu().numpy(),
                zmin=0,
                zmax=1,
                colorscale=[[0, "black"], [1, "white"]],
                showscale=False,
            ),
            row=row + 1,
            col=col,
        )
fig.update_layout(
    title="Flow ODE sampling: black = p(0), white = p(1)",
    height=850,
    width=1900,
)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False, autorange="reversed")
fig.show()

solved = [solvability_rate([maze]) for maze in hard_samples]
print(f"Solvable samples: {sum(solved)}/{len(solved)} ({np.mean(solved):.1%})")
hard_fig = make_subplots(
    rows=1,
    cols=N_TO_SHOW,
    subplot_titles=[
        f"Sample {i + 1}: {'solvable' if solved[i] else 'unsolvable'}"
        for i in range(N_TO_SHOW)
    ],
)
for i in range(N_TO_SHOW):
    hard_fig.add_trace(
        go.Heatmap(
            z=hard_samples[i],
            zmin=0,
            zmax=1,
            colorscale=[[0, "black"], [1, "white"]],
            showscale=False,
        ),
        row=1,
        col=i + 1,
    )
hard_fig.update_layout(title="Final mazes after greedy decoding", height=390, width=900)
hard_fig.update_xaxes(showticklabels=False)
hard_fig.update_yaxes(showticklabels=False, autorange="reversed")
hard_fig.show()

Initial Gaussian noise stats: mean=0.001, std=1.001


Solvable samples: 26.0/128 (20.3%)
